# TWAS, Colocalization & ColocBoost

This notebook demonstrates four downstream analyses that integrate molecular QTLs with GWAS, using the toy `protocol_example` dataset. Each method is a self-contained part with its own input files and output files.

## Pipeline position & prerequisites

This is the final, **advanced-analysis** notebook in the `protocol_example` toy bundle. It consumes artifacts produced by earlier notebooks, so run them first in this order:

1. `data_preprocessing/2_genotype_preprocessing` → produces `output/plink/` (QC'd genotypes).
2. `data_preprocessing/1_phenotype_preprocessing` → produces `output/phenotype/` (molecular phenotypes by chrom).
3. `data_preprocessing/4_covariates_preprocessing` → produces `output/covariate/` (covariates + PCs).
4. `qtl_association_finemapping/1_xqtl_association` → produces `output/phenotype/phenotype_by_chrom_for_cis/` region lists.
5. `qtl_association_finemapping/2_finemapping` → produces `output/finemapping/susie_twas/twas_weights/` (the precomputed TWAS prediction weights Part 1 needs).

Within this notebook the parts also chain to each other: earlier parts write to `output/twas/`, `output/susie_coloc/`, and `output/colocboost/`, which later parts read back.

**Pre-staged external inputs.** A few reference inputs are not produced by any notebook in this bundle, so they are provided ready-made under `input/`: `input/twas/` (TWAS meta/LD-block tables), `input/LD_sketch/` (RSS LD sketch reference), `input/susie_enloc_data/` (enloc priors & GWAS blocks), and `input/colocboost/` (ColocBoost GWAS meta). The specific input files each part needs are listed at the start of that part.


## Overview

1. **TWAS / MR** — transcriptome-wide association and Mendelian randomization.
2. **xQTL–GWAS enrichment** — global enrichment parameters used as colocalization priors.
3. **Pairwise colocalization (SuSiE-coloc)** — gene-by-gene xQTL–GWAS colocalization.
4. **ColocBoost** — multi-trait colocalization (xQTL-only and joint xQTL–GWAS modes).

The input files needed for each part are listed at the start of that part.

## 1. TWAS

**TWAS** tests gene–trait associations by combining precomputed molecular phenotype prediction weights with GWAS summary statistics, followed by Mendelian Randomization (MR) on candidate genes.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/twas/protocol_example.twas.gwas_meta.tsv` | toy data prep | GWAS summary-statistics metadata |
| `input/twas/protocol_example.twas.xqtl_meta.tsv` | toy data prep | xQTL weight metadata (gene region + RDS weights) |
| `input/twas/protocol_example.twas.LD_blocks.chr22.bed` | toy data prep | analysis region definitions |
| `input/twas/protocol_example.twas.data_type_table.txt` | toy data prep | xQTL molecular phenotype type table |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata |
| `output/finemapping/susie_twas/twas_weights/*.univariate_twas_weights.rds` | fine-mapping (2_finemapping) | TWAS prediction weights |

Key parameters: `--rsq_cutoff 0.01` (cross-validation R² threshold for predictable genes) and `--rsq_pval_cutoff` (cross-validation p-value threshold).

The TWAS step proceeds in four stages. First it **loads** the precomputed prediction weights from the RDS files. It then identifies **predictable genes** — those whose expression is reliably imputable based on cross-validation performance (controlled by `--rsq_cutoff` and `--rsq_pval_cutoff`). For those genes it computes **TWAS Z-scores** by combining the predicted expression with the GWAS summary statistics, and finally performs **Mendelian Randomization (MR)** on the candidate genes to assess potential causal effects of expression on the trait.

### **Step 1.** Load the TWAS prediction weights


In [16]:
# gwas input 
cd /home/ubuntu/xqtl_protocol_exercise
head data/twas/gwas_meta_test.tsv

List of 1
 $ ENSG00000283047:List of 1
  ..$ bulk_rnaseq_ENSG00000283047:List of 9


In [17]:
# ld meta file
head reference_data/ADSP_R4_EUR/ld_meta_file.tsv

 NULL


### **Step 2.** Run TWAS and Mendelian Randomization


In [ ]:
sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output/twas \
    --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --ld_meta_data input/LD_sketch/meta.tsv \
    --ld_reference_sample_size 60 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --rsq_pval_cutoff 1 \
    --rsq_cutoff 0.01

In [ ]:
# twas weight file
head data/twas/mwe_twas_pipeline_test_small.tsv

### Inspect the TWAS and MR results


In [ ]:
# change this kernel to R code
# TWAS weights structure
setwd('/home/ubuntu/xqtl_protocol_exercise')
twas_weight = readRDS('data/twas/ROSMAP_DeJager.chr11_ENSG00000002330.univariate_twas_weights.rds')
str(twas_weight)

In [ ]:
# r code
twas_weight_by_method = twas_weight$ENSG00000002330$AC_DeJager_eQTL_ENSG00000002330$twas_weights
str(twas_weight_by_method)

### Output Files

| File | Contents |
|------|----------|
| `output/twas/twas/protocol_example.chr22_*.twas.tsv.gz` | Per-gene TWAS Z-scores and p-values |
| `output/twas/twas/protocol_example.chr22_*.mr_result.tsv.gz` | Mendelian Randomization results for candidate genes |

## 2. xQTL–GWAS enrichment

Estimates global enrichment parameters (a0, a1) between xQTL and GWAS signals and converts them into colocalization priors (p1, p2, p12) used by the colocalization steps that follow.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |

### **Step 1.** Run xQTL–GWAS enrichment analysis


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb xqtl_gwas_enrichment \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --cwd output/xqtl_gwas_enrichment


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/twas_ctwas.ipynb twas \
   --cwd output/twas --name test \
   --gwas_meta_data data/twas/gwas_meta_test.tsv \
   --ld_meta_data reference_data/ADSP_R4_EUR/ld_meta_file.tsv \
   --ld_reference_sample_size 17000 \
   --regions data/twas/EUR_LD_blocks.bed \
   --xqtl_meta_data data/twas/mwe_twas_pipeline_test_small.tsv \
   --xqtl_type_table data/twas/data_type_table.txt \
   --rsq_pval_cutoff 0.05 --rsq_cutoff 0.01    

### Inspect the enrichment parameters


In [ ]:
## twas results
cd /home/ubuntu/xqtl_protocol_exercise
zcat output/twas/twas/test.chr11_84267999_86714492.twas.tsv.gz | head -n 4

### Output Files

| File | Contents |
|------|----------|
| `output/xqtl_gwas_enrichment/*.enrichment.rds` | Enrichment parameters (a0, a1) and derived colocalization priors (p1, p2, p12) |

## 3. Pairwise colocalization (SuSiE-coloc)

Runs gene-by-gene xQTL–GWAS colocalization with `susie_coloc`, using either the enrichment-derived priors from Part 2 or default priors (`--skip-enrich`). LD metadata enables variant-level credible sets.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata (with overlap info) |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/ld_reference/protocol_example.ld_meta_file.tsv` | toy data prep | LD reference metadata for credible sets |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |

### **Step 1.** Run pairwise colocalization (SuSiE-coloc)


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb susie_coloc \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --skip-enrich \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --ld_meta_file_path input/ld_reference/protocol_example.ld_meta_file.tsv \
    --cwd output/susie_coloc


In [ ]:
## mr results
zcat output/twas/twas/test.chr11_84267999_86714492.mr_result.tsv.gz | head

### Inspect the colocalization results


### Output Files

| File | Contents |
|------|----------|
| `output/susie_coloc/susie_coloc/*@<gene>.coloc.rds` | Per gene-context colocalization stats and posterior probabilities |
| `output/susie_coloc/*.coloc_res` | Variant-level credible-set results (when LD metadata is supplied) |

Each `.coloc.rds` is a nested R list with `summary` (posterior probabilities for the five coloc hypotheses), `results` (SNP-level PP.H4), `priors`, and `analysis_region`. **PP.H4** is the probability that both traits share a single causal variant — higher values mean stronger colocalization evidence.

Each `.coloc.rds` reports posterior probabilities for five mutually exclusive colocalization hypotheses for a given gene-trait pair: **PP.H0** (neither trait has an association in the region), **PP.H1** (only the xQTL is associated), **PP.H2** (only the GWAS trait is associated), **PP.H3** (both are associated but driven by *different* causal variants), and **PP.H4** (both are associated and share a *single* causal variant). A high **PP.H4** is the evidence for colocalization — it indicates the molecular and complex traits are likely driven by the same underlying variant.

## 4. ColocBoost

Multi-trait colocalization that jointly analyzes molecular traits, optionally together with GWAS. Two modes are shown: **xQTL-only** (`--no-separate-gwas`) and **joint xQTL–GWAS** (`--separate-gwas`).

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `output/plink/protocol_example.genotype.merged.plink_qc.bed` | genotype preprocessing | merged genotypes |
| `output/phenotype/.../bulk_rnaseq.phenotype_by_chrom_files.region_list.txt` | phenotype preprocessing | phenotype region list |
| `output/covariate/protocol_example...Marchenko_PC.gz` | covariate preprocessing | covariates |
| `input/reference_data/TAD/TADB_enhanced_cis.bed` | reference data | cis association windows |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata (joint mode) |
| `input/colocboost/gwas_meta.txt` | toy data prep | GWAS metadata, tab-separated (joint mode) |

### **Step 1.** xQTL-only ColocBoost (`--no-separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name test_coloc_boost_xqtl_only \
    --cwd output/colocboost \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --no-separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


In [ ]:
# ouput of enrichment parameter between xqtl and gwas
# R COED
setwd('/home/ubuntu/xqtl_protocol_exercise')
coloc_enrichment = readRDS('output/xqtl_gwas_enrichment/demo_overlap.overlapped.demo_gwas.Knight_eQTL_brain.enrichment.rds')
str(coloc_enrichment)

### **Step 2.** Joint xQTL–GWAS ColocBoost (`--separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name colocboost_gwas \
    --cwd output/colocboost_gwas \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --ld-meta-data input/LD_sketch/meta.tsv \
    --gwas-meta-data input/colocboost/gwas_meta.txt \
    --separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


In [ ]:
# gwas meta
cd /home/ubuntu/xqtl_protocol_exercise
head data/susie_enloc_data/demo_gwas.block_results_db.tsv

### Inspect the ColocBoost output


### Output Files

| File | Contents |
|------|----------|
| `output/colocboost/.../cb_xqtl_*.rds` | xQTL-only ColocBoost result per gene |
| `output/colocboost_gwas/.../cb_xqtl_synthetic_gwas_*.rds` | Joint xQTL–GWAS ColocBoost result per gene |

Each `.rds` is a named list (class `colocboost`) keyed by gene. The most useful elements are `cos_summary` (colocalized signals, empty if none), `cos_details` (details per colocalized signal), `ucos_details` (uncolocalized signals — many of these suggest GWAS signals independent of molecular QTLs), and `data_info` / `model_info` (traits analyzed and model convergence).

In [ ]:
# r code
# coloc results
setwd('/home/ubuntu/xqtl_protocol_exercise')
coloc_result = readRDS('output/susie_coloc/susie_coloc/demo_overlap.overlapped.demo_gwas.Knight_eQTL_brain@ENSG00000142798.coloc.rds')
str(coloc_result)

List of 1
 $ ENSG00000142798:List of 5
  ..$ summary        :Classes ‘data.table’ and 'data.frame':	10 obs. of  10 variables:
  .. ..$ nsnps    : int [1:10] 8826 8826 8826 8826 8826 8826 8826 8826 8826 8826
  .. ..$ hit1     : chr [1:10] "1:21912282:C:G" "1:21912282:C:G" "1:21912282:C:G" "1:21912282:C:G" ...
  .. ..$ hit2     : chr [1:10] "1:23996556:C:T" "1:22187644:C:A" "1:22187644:C:A" "1:22187644:C:A" ...
  .. ..$ PP.H0.abf: num [1:10] 0.0165 0.0716 0.0715 0.0715 0.0715 ...
  .. ..$ PP.H1.abf: num [1:10] 0.102 0.443 0.443 0.443 0.443 ...
  .. ..$ PP.H2.abf: num [1:10] 0.1217 0.0645 0.0645 0.0645 0.0645 ...
  .. ..$ PP.H3.abf: num [1:10] 0.753 0.399 0.399 0.399 0.399 ...
  .. ..$ PP.H4.abf: num [1:10] 0.00699 0.02221 0.0222 0.0222 0.0222 ...
  .. ..$ idx1     : int [1:10] 1 1 1 1 1 1 1 1 1 1
  .. ..$ idx2     : int [1:10] 1 2 3 4 5 6 7 8 9 10
  .. ..- attr(*, ".internal.selfref")=<externalptr> 
  ..$ results        :Classes ‘data.table’ and 'data.frame':	8826 obs. of  11 variables:


In [ ]:
# xQTL-only with --no-separate-gwas parameter
sos run pipeline/colocboost.ipynb colocboost \
    --name test_coloc_boost_xqtl_only  \
    --cwd output/colocboost \
    --genoFile output/plink/wgs.merged.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --no-separate-gwas --xqtl-coloc \
    --region-name ENSG00000049246 \
    --phenotype-names trait_A

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO: NumExpr defaulting to 16 threads.
INFO: Running get_rss_analysis_regions: 
INFO: get_rss_analysis_regions is completed.
INFO: get_rss_analysis_regions output:   regional_rss_data
INFO: Running get_analysis_regions: 
Loading customized association analysis window from reference_data/TAD/TADB_enhanced_cis.bed
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: Running colocboost: 
INFO: colocboost is completed.
INFO: colocboost output:   /mnt/vast/hpc/homes/al4225/xqtl_protocol_data/output/colocbo

In [ ]:
# GWAS-xQTL integrated with --separate-gwas parameter
sos run pipeline/colocboost.ipynb colocboost \
    --name colocboost_gwas  \
    --cwd output/colocboost_gwas \
    --genoFile output/plink/wgs.merged.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --ld-meta-data data/ld_meta_file.tsv \
    --gwas-meta-data data/colocboost/gwas_meta.txt \
    --separate-gwas --xqtl-coloc \
    --region-name ENSG00000049246 \
    --phenotype-names trait_A

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO: NumExpr defaulting to 16 threads.
INFO: Running get_analysis_regions: 
Loading customized association analysis window from reference_data/TAD/TADB_enhanced_cis.bed
INFO: Running get_rss_analysis_regions: 
Using customized association window file: reference_data/TAD/TADB_enhanced_cis.bed
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: get_rss_analysis_regions is completed.
INFO: get_rss_analysis_regions output:   regional_rss_data
INFO: Running colocboost: 
INFO: colocboost is completed.
INFO

In [ ]:
# output of colocboost
setwd('/home/ubuntu/xqtl_protocol_exercise')
colocboost = readRDS('output/colocboost_gwas/colocboost/colocboost_gwas.chr1_ENSG00000049246.cb_xqtl_Wightman.rds')
str(colocboost)

List of 1
 $ ENSG00000049246:List of 8
  ..$ cos_summary   : NULL
  ..$ vcp           : NULL
  ..$ cos_details   : NULL
  ..$ data_info     :List of 6
  .. ..$ n_outcomes  : int 2
  .. ..$ n_variables : int 6872
  .. ..$ outcome_info:'data.frame':	2 obs. of  4 variables:
  .. .. ..$ outcome_names: chr [1:2] "trait_A_ENSG00000049246" "Wightman"
  .. .. ..$ sample_size  : num [1:2] 150 363646
  .. .. ..$ is_sumstats  : logi [1:2] FALSE TRUE
  .. .. ..$ is_focal     : logi [1:2] FALSE TRUE
  .. ..$ variables   : chr [1:6872] "chr1:6120249:G:A" "chr1:6120384:G:A" "chr1:6120435:G:A" "chr1:6120926:G:A" ...
  .. ..$ coef        :List of 2
  .. .. ..$ trait_A_ENSG00000049246: num [1:6872] 0 0 0 0 0 0 0 0 0 0 ...
  .. .. ..$ Wightman               : num [1:6872] -1.30e-05 -1.15e-05 -1.18e-05 1.10e-05 -1.31e-05 ...
  .. ..$ z           :List of 2
  .. .. ..$ trait_A_ENSG00000049246: num [1:6872] 0 0 0 0 0 0 0 0 0 0 ...
  .. .. ..$ Wightman               : num [1:6872] -1.0368 -0.2813 -0.4772 0.0